In [0]:
from pyspark.sql import functions as F

# Parámetro del Job: cuántos días nuevos aterrizar en esta corrida
dbutils.widgets.text("n_dias_a_aterrizar", "20")
N_DIAS = int(dbutils.widgets.get("n_dias_a_aterrizar"))

CATALOG = "workspace"
SCHEMA = "nyctaxi_project"
LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing/trips"

source_df = (
    spark.table("samples.nyctaxi.trips")
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
)

dates = [r["pickup_date"] for r in
         source_df.select("pickup_date").distinct().orderBy("pickup_date").collect()]

# Qué días ya están aterrizados
try:
    ya_aterrizados = {f.name.rstrip("/") for f in dbutils.fs.ls(LANDING_PATH)}
except Exception:
    ya_aterrizados = set()

pendientes = [d for d in dates if str(d) not in ya_aterrizados]
a_aterrizar = pendientes[:N_DIAS]

def land_day(day):
    (source_df
        .filter(F.col("pickup_date") == F.lit(day))
        .drop("pickup_date")
        .coalesce(1)
        .write.mode("overwrite")
        .option("header", True)
        .csv(f"{LANDING_PATH}/{day}"))

for d in a_aterrizar:
    land_day(d)

print(f"Ya aterrizados: {len(ya_aterrizados)} | Aterrizados ahora: {len(a_aterrizar)} | Pendientes: {len(pendientes) - len(a_aterrizar)}")

Ya aterrizados: 60 | Aterrizados ahora: 0 | Pendientes: 0
